# 05 Feature Engineering\nPurpose: Build model-ready features and create baseline default scoring artifacts.

In [ ]:
from pathlib import Path\nimport pandas as pd\nimport numpy as np\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import roc_auc_score\n\nROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd()\nCLEAN_PATH = ROOT / 'data' / 'processed' / 'loan_clean.csv'\nFEATURE_OUT = ROOT / 'data' / 'processed' / '05_feature_matrix.parquet'\nSCORE_OUT = ROOT / 'data' / 'processed' / '05_default_scored.csv'\nIMPORTANCE_OUT = ROOT / 'reports' / 'phase_05_feature_strength.csv'

In [ ]:
df = pd.read_csv(CLEAN_PATH)\ndf['LoanToIncome'] = (df['LoanAmount'] / df['Income']).replace([np.inf, -np.inf], np.nan).fillna(0)\ndf['RateXDTI'] = df['InterestRate'] * df['DTIRatio']\ndf['IsThinFile'] = (df['NumCreditLines'] <= 1).astype(int)\ndf[['LoanToIncome','RateXDTI','IsThinFile']].head()

In [ ]:
target = 'Default'\nfeatures = [c for c in df.columns if c not in ['LoanID', target]]\nX = pd.get_dummies(df[features], drop_first=True)\ny = df[target]\nloan_ids = df['LoanID']\n\nX_train, X_test, y_train, y_test, id_train, id_test = train_test_split(\n    X, y, loan_ids, test_size=0.25, random_state=42, stratify=y\n)\nmodel = LogisticRegression(max_iter=500, class_weight='balanced')\nmodel.fit(X_train, y_train)\npred_prob = model.predict_proba(X_test)[:, 1]\nauc = roc_auc_score(y_test, pred_prob)\nprint({'test_auc': round(float(auc), 4), 'test_rows': int(len(X_test))})

In [ ]:
feature_strength = pd.DataFrame({\n    'feature': X.columns,\n    'coefficient': model.coef_.flatten()\n})\nfeature_strength['abs_coefficient'] = feature_strength['coefficient'].abs()\nfeature_strength = feature_strength.sort_values('abs_coefficient', ascending=False)\nfeature_strength.to_csv(IMPORTANCE_OUT, index=False)\nfeature_strength.head(20)

In [ ]:
scored = pd.DataFrame({\n    'LoanID': id_test.values,\n    'actual_default': y_test.values,\n    'default_probability': pred_prob,\n})\nscored['risk_band'] = pd.qcut(scored['default_probability'], q=5, labels=['Very Low','Low','Medium','High','Very High'])\nscored.to_csv(SCORE_OUT, index=False)\nX.to_parquet(FEATURE_OUT, index=False)\nprint('saved:', SCORE_OUT)\nprint('saved:', FEATURE_OUT)